<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/%EB%B0%98%EB%8F%84%EC%B2%B4_FDC_%EC%9D%B4%EC%83%81_%ED%83%90%EC%A7%80_%EC%A0%84%EC%B2%B4_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8_R03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import time

# matplotlib 설정 (스타일 지정, 한글 폰트 배제)
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

print("=== Phase 1: Realistic FDC Trace Data Generation ===")

# 1. 시뮬레이션 환경 설정
num_wafers = 200     # 웨이퍼 수 증가
seq_len = 200        # 총 200초 (10개 스텝 x 20초)
num_params = 10      # 10개의 핵심 파라미터
num_steps = 10       # 10개의 공정 스텝

params_list = [
    'Pressure', 'RF_Top', 'RF_Btm', 'Gas_Cl2', 'Gas_BCl3',
    'Gas_Ar', 'Temp_Wall', 'Temp_ESC', 'VPP', 'APC_Angle'
]

step_names = [f'Step{i+1}' for i in range(num_steps)]

# 2. 고속 텐서 데이터 생성
tensor_data = np.zeros((num_wafers, seq_len, num_params))
scenario_labels = ['Normal'] * num_wafers
np.random.seed(42)

start_time = time.time()
for w in range(num_wafers):
    # 장비 노후화 (Drift) 계수
    drift_factor = (w / num_wafers)

    for t in range(seq_len):
        step_idx = t // 20  # 20초마다 스텝 변경

        # [기본 베이스라인 로직] 스텝별 파라미터 변동
        if 2 <= step_idx <= 6: # Main Etch 구간 (Step 3 ~ 7)
            base_vals = [50.0, 1000.0, 300.0, 150.0, 50.0, 200.0, 60.0, 20.0, 800.0, 45.0]
        else: # Idle or Purge 구간
            base_vals = [5.0, 0.0, 0.0, 0.0, 0.0, 500.0, 55.0, 20.0, 0.0, 85.0]

        # 센서별 고유 진동(Sine) 및 노이즈 추가
        pressure = base_vals[0] + np.sin(t * 0.5) * 2 + np.random.normal(0, 0.5)
        rf_top   = base_vals[1] + np.random.normal(0, 5.0) if base_vals[1] > 0 else 0
        rf_btm   = base_vals[2] + np.random.normal(0, 2.0) if base_vals[2] > 0 else 0
        gas_cl2  = base_vals[3] + np.random.normal(0, 1.0)
        gas_bcl3 = base_vals[4] + np.random.normal(0, 0.5)
        gas_ar   = base_vals[5] + np.random.normal(0, 2.0)
        temp_wall= base_vals[6] + drift_factor * 5.0 + np.random.normal(0, 0.1) # 서서히 온도 상승
        temp_esc = base_vals[7] + np.random.normal(0, 0.05)
        vpp      = base_vals[8] + np.random.normal(0, 10.0) if base_vals[8] > 0 else 0
        apc_angle= base_vals[9] - np.sin(t * 0.5) * 1.5 + np.random.normal(0, 0.5) # 기압과 반대로 움직임

        # --- 이상 시나리오 주입 ---
        # 시나리오 1: Step Drift (Wafer 140~160, Main Etch 구간)
        if 140 <= w < 160 and 2 <= step_idx <= 6:
            scenario_labels[w] = 'Scenario1_Drift'
            abnorm_drift = ((w - 140) / 20) * 20.0 # 기압 상승
            pressure += abnorm_drift
            apc_angle -= abnorm_drift * 0.5        # 기압을 낮추기 위해 APC 밸브가 더 열림(각도 감소)

        # 시나리오 2: Sudden Arcing (Wafer 170~200 중 무작위 5개, 특정 초)
        if w in [172, 178, 185, 192, 198] and 2 <= step_idx <= 6:
            scenario_labels[w] = 'Scenario2_Arcing'
            # 85~88초 사이에 아킹 발생
            if 85 <= t <= 88:
                rf_top += 400.0
                vpp += 300.0

        tensor_data[w, t, :] = [pressure, rf_top, rf_btm, gas_cl2, gas_bcl3,
                                gas_ar, temp_wall, temp_esc, vpp, apc_angle]

print(f"Data Generation Complete. Shape: {tensor_data.shape} ({time.time() - start_time:.2f} sec)")

# 3. CSV 변환 및 저장 (Long Format)
print("Converting to Long Format and saving to CSV...")
long_format_data = []
for w in range(num_wafers):
    w_id = f'WFR-{w+1:03d}'
    label = scenario_labels[w]
    for t in range(seq_len):
        step_id = step_names[t // 20]
        timestamp = f'2026-04-01 10:{t//60:02d}:{t%60:02d}'

        for p_idx, p_name in enumerate(params_list):
            long_format_data.append([
                timestamp, w_id, step_id, p_name, tensor_data[w, t, p_idx], label
            ])

df_raw = pd.DataFrame(long_format_data, columns=['TIMESTAMP', 'WAFER_ID', 'STEP_ID', 'PARA_NAME', 'VALUE', 'SCENARIO_LABEL'])
df_raw.to_csv('realistic_fdc_trace.csv', index=False)
print("Saved to 'realistic_fdc_trace.csv'.")


print("\n=== Phase 2: Data Preprocessing (Normalization) ===")
# 훈련 데이터: Wafer 1 ~ 100 (정상 모델용)
train_data = tensor_data[:100]

mean_val = train_data.mean(axis=(0, 1))
std_val = train_data.std(axis=(0, 1))

def normalize(data):
    return (data - mean_val) / (std_val + 1e-7)

def denormalize(data):
    return data * (std_val + 1e-7) + mean_val

tensor_data_norm = normalize(tensor_data)
train_data_norm = normalize(train_data)


print("\n=== Phase 3: High-Performance Model Building (Multi-GPU setup) ===")
class TS_MAE_Large(nn.Module):
    def __init__(self, num_features, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        self.input_proj = nn.Linear(num_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, dim_feedforward=2048, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, num_features)

    def forward(self, x, mask_ratio=0.15):
        batch_size, seq_len, _ = x.shape
        if mask_ratio > 0.0 and self.training:
            mask = torch.rand(batch_size, seq_len).to(x.device) < mask_ratio
            x_masked = x.clone()
            x_masked[mask] = 0.0
        else:
            x_masked = x

        encoded = self.input_proj(x_masked)
        out = self.transformer(encoded)
        return self.output_proj(out)

# GPU 할당 (P100 4대 활용)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TS_MAE_Large(num_features=num_params).to(device)

if torch.cuda.device_count() > 1:
    print(f"Utilizing {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)
else:
    print("Running on single GPU or CPU.")

optimizer = optim.Adam(model.parameters(), lr=0.0005) # Large model = slightly lower LR
criterion = nn.MSELoss()

# 모델 학습
train_tensor = torch.FloatTensor(train_data_norm).to(device)
epochs = 100
batch_size = 32 # GPU 메모리에 맞춰 큰 배치 사용 가능

model.train()
print("Starting Pre-training on Golden Wafers...")
loss_history = []

for epoch in range(epochs):
    epoch_loss = 0
    # 배치 셔플링
    indices = torch.randperm(len(train_tensor))
    train_tensor_shuffled = train_tensor[indices]

    for i in range(0, len(train_tensor), batch_size):
        batch = train_tensor_shuffled[i:i+batch_size]

        optimizer.zero_grad()
        recon = model(batch, mask_ratio=0.25) # 25% 마스킹으로 난이도 상승
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / (len(train_tensor)/batch_size)
    loss_history.append(avg_loss)
    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], MSE Loss: {avg_loss:.4f}")


print("\n=== Phase 4: Unsupervised Inference & Scatter Plot ===")
def calculate_anomaly_scores(model, data_tensor):
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(data_tensor).to(device)
        recon = model(x, mask_ratio=0.0)
        error_map = (x - recon) ** 2
        wafer_scores = torch.mean(error_map, dim=(1, 2)).cpu().numpy()
    return wafer_scores, error_map.cpu().numpy(), recon.cpu().numpy()

all_data_tensor = tensor_data_norm
anomaly_scores, error_maps, recon_data_norm = calculate_anomaly_scores(model, all_data_tensor)

# 동적 임계치(Threshold) 설정: 훈련 데이터(1~100)의 복원 오차 최대값의 1.5배로 가정 (라벨 미사용)
train_scores = anomaly_scores[:100]
threshold = np.max(train_scores) * 1.5

wafer_indices = np.arange(1, 201)
is_anomaly = anomaly_scores > threshold

# [Visualization] Scatter Plot (Unsupervised Detection)
plt.figure(figsize=(14, 6))

# 정상으로 판별된 웨이퍼
plt.scatter(wafer_indices[~is_anomaly], anomaly_scores[~is_anomaly],
            color='dodgerblue', alpha=0.6, label='Predicted Normal')
# 이상으로 판별된 웨이퍼
plt.scatter(wafer_indices[is_anomaly], anomaly_scores[is_anomaly],
            color='red', marker='x', s=80, linewidth=2, label='Predicted Anomaly')

plt.axhline(y=threshold, color='black', linestyle='--', label=f'Detection Threshold ({threshold:.4f})')

plt.title('Unsupervised Equipment Health Monitoring: Wafer Anomaly Scatter', fontsize=16)
plt.xlabel('Wafer Processing Sequence', fontsize=12)
plt.ylabel('Anomaly Score (Reconstruction Error)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.4)
plt.show()


print("\n=== Phase 5: Auto-Triggered Root Cause Analysis Report ===")
def generate_business_report(target_idx):
    w_id = f'WFR-{target_idx+1:03d}'
    score = anomaly_scores[target_idx]
    e_map = error_maps[target_idx]

    print(f"\n[RCA Report Generated for {w_id}] - Score: {score:.4f}")

    # 1. Heatmap (X-ray for Root Cause) - 가독성을 위해 크기 대폭 축소
    plt.figure(figsize=(10, 3))
    sns.heatmap(e_map.T, cmap='Reds', cbar_kws={'label': 'Error Magnitude'})
    plt.yticks(ticks=np.arange(num_params)+0.5, labels=params_list, rotation=0, fontsize=9)
    plt.title(f'Root Cause Analysis X-Ray (Target: {w_id} - Auto Detected)', fontsize=12)
    plt.xlabel('Process Time (sec) / Step Progression', fontsize=10)

    for i in range(1, num_steps):
        plt.axvline(i * 20, color='blue', linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # 2. Detailed Trace Comparison (전체 10개 파라미터 5x2 그리드로 깔끔하게 표시)
    actual_trace = denormalize(all_data_tensor[target_idx])
    recon_trace = denormalize(recon_data_norm[target_idx])

    fig, axes = plt.subplots(5, 2, figsize=(14, 10), sharex=True)
    axes = axes.flatten()

    for p_idx, param_name in enumerate(params_list):
        axes[p_idx].plot(actual_trace[:, p_idx], label='Actual Sensor Value', color='black', linewidth=1.0)
        axes[p_idx].plot(recon_trace[:, p_idx], label='AI Expected Value', color='red', linestyle='--', linewidth=1.0)

        diff = np.abs(actual_trace[:, p_idx] - recon_trace[:, p_idx])
        dynamic_threshold = np.std(actual_trace[:, p_idx]) * 0.5
        axes[p_idx].fill_between(range(seq_len), 0, 1, where=(diff > dynamic_threshold),
                             color='red', alpha=0.2, transform=axes[p_idx].get_xaxis_transform())

        axes[p_idx].set_title(param_name, fontsize=10, pad=3)
        if p_idx == 1: # 우상단 그래프에만 범례를 표시하여 나머지 그래프의 공간 확보
            axes[p_idx].legend(loc='upper right', fontsize=8)
        for j in range(1, num_steps):
            axes[p_idx].axvline(j * 20, color='gray', linestyle=':', alpha=0.3)

    # 하단 두 개(8, 9번)의 x축에만 라벨 표시
    for i in range(8, 10):
        axes[i].set_xlabel('Process Time (sec)', fontsize=10)

    plt.suptitle(f'Detailed Sensor Analysis: Actual vs AI Expectation ({w_id})', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()

# 임계치를 초과한 (이상으로 감지된) 웨이퍼 인덱스 추출
detected_anomalies = np.where(is_anomaly)[0]
print(f"Total Anomalies Detected: {len(detected_anomalies)}")

# 리포트 자동 생성: 검출된 모든 웨이퍼에 대해 점수가 '가장 높은 순'으로 전체 출력
if len(detected_anomalies) > 0:
    sorted_anomalies = detected_anomalies[np.argsort(anomaly_scores[detected_anomalies])[::-1]]

    print(f"Generating RCA reports for ALL {len(sorted_anomalies)} anomalies in descending order of severity...")
    for target_idx in sorted_anomalies:
        generate_business_report(target_idx)
else:
    print("No anomalies detected above the given threshold.")

print("\n=== Phase 6: Future Trend Prediction (Prognostics) ===")
# 인덱스 기반 미래 건강도 추세선 예측 (Polynomial Curve Fitting)
recent_window = 100 # 최근 100개 웨이퍼 데이터를 기반으로 미래 예측
recent_wafers = wafer_indices[-recent_window:]
recent_scores = anomaly_scores[-recent_window:]

# 노이즈를 줄이기 위해 이동 평균(Moving Average) 적용
smoothed_scores = pd.Series(recent_scores).rolling(window=5, min_periods=1).mean().values

# 2차 다항식 피팅 (장비의 가속화되는 노후화 곡선 반영)
poly_coeffs = np.polyfit(recent_wafers, smoothed_scores, 2)
poly_func = np.poly1d(poly_coeffs)

# 향후 50 웨이퍼 스케줄에 대한 고장 위험도 예측
future_wafers = np.arange(num_wafers + 1, num_wafers + 51)
future_predictions = poly_func(future_wafers)

plt.figure(figsize=(14, 6))

# 기존 데이터 (흐리게 배경 처리하여 추세 강조)
plt.scatter(wafer_indices, anomaly_scores, color='lightgray', s=20, label='Actual Anomaly Scores')
plt.plot(recent_wafers, smoothed_scores, color='blue', linewidth=2, label='Smoothed Trend (Recent 100 Wafers)')

# 미래 예측 선
plt.plot(future_wafers, future_predictions, color='red', linestyle='--', linewidth=2.5, label='Predicted Future Trend')

# 임계치 라인
plt.axhline(y=threshold, color='black', linestyle='--', label=f'Warning Threshold ({threshold:.2f})')
critical_threshold = threshold * 2.0  # 설비 중단이 필요한 한계선 가이드
plt.axhline(y=critical_threshold, color='darkred', linestyle='-', label=f'Critical Failure Limit ({critical_threshold:.2f})')

plt.title('Equipment Prognostics: Future Anomaly Trend Prediction', fontsize=16)
plt.xlabel('Wafer Processing Sequence (Including Future)', fontsize=12)
plt.ylabel('Anomaly Score', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()